# автоматические главы и тематическая навигация для подкастов и интервью

групповой проект по курсу. мы делаем MVP для продюсерской студии: на вход даём запись выпуска, на выходе получаем расшифровку, главы с таймкодами, темы, спикеров и готовое описание для YouTube. продюсер всё это проверяет и правит, а не пишет с нуля.

Полный эксперимент выполнен в Colab с GPU Tesla T4. Завершились все 24 ячейки кода и 40 автоматических тестов. Выводы сверены с архивом этого прогона.

Это публичная версия для повторного запуска. Модули загружаются из каталога source/ репозитория, авторская разметка — с сайта «Радио-Т». Таблицы, графики и материалы защиты собраны в [results](https://github.com/majlychko/Group_Homework/tree/main/results). При выполнении вы получите выводы ячеек в своей среде.


In [ ]:
#@title 1. Параметры запуска
RUN_PROFILE = "full" #@param ["full", "quick"]
DEVICE = "cuda" #@param ["cuda", "cpu"]
RUN_STRESS_TEST = True #@param {type:"boolean"}
DEMO_SOURCE = "" #@param {type:"string"}
DEMO_LIMIT_MIN = 0 #@param {type:"integer"}
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
REPOSITORY_URL = "https://github.com/majlychko/Group_Homework" #@param {type:"string"}

# full - мы берём все 6 выпусков целиком, quick - 2 dev и 2 test выпуска по 60 минут
EVAL_LIMIT_MIN = 0 if RUN_PROFILE == "full" else 60
# у каждого профиля своя папка, чтобы быстрый прогон не перемешался с полным
WORK_DIRECTORY = "podcast_runs/" + RUN_PROFILE
# пустой DEMO_SOURCE значит, что в демо мы показываем готовый результат первого test-выпуска
# проверяем, что в форме не ввели что-то невозможное
assert RUN_PROFILE in {"full", "quick"}
assert DEMO_LIMIT_MIN >= 0
print("Профиль:", RUN_PROFILE, "| лимит минут:", EVAL_LIMIT_MIN or "выпуск целиком")

In [ ]:
# ставим пакеты до первого импорта моделей, поэтому нам нужна свежая среда выполнения
import sys, subprocess, os
from pathlib import Path
# версии закрепляем, чтобы прогон повторялся; torch с cuda берём тот, что уже есть в образе colab
# gigaam-v3 собирается через hydra и omegaconf, поэтому ставим их заранее
requirements = '# ставим до импорта torch и transformers, torch и torchaudio с cuda оставляем из образа colab\n# silero-vad ставится отдельно с --no-deps, чтобы он не потянул другую версию torch\ntransformers==5.23.0\n# gigaam-v3 собирает модель через hydra и omegaconf, токенизатор e2e-версии на sentencepiece\nhydra-core==1.3.8\nomegaconf==2.3.0\nnumpy==2.2.6\npandas==2.2.3\nscipy==1.15.3\nscikit-learn==1.6.1\nmatplotlib==3.10.3\nsoundfile==0.13.1\nlibrosa==0.11.0\naccelerate==1.11.0\nrequests==2.32.5\nsentencepiece==0.2.1\nyt-dlp==2026.10.26\nipywidgets==8.1.7\n# для youtube иногда нужен более свежий yt-dlp, основной эксперимент от youtube не зависит\n'
Path("requirements-podcast-colab.txt").write_text(requirements, encoding="utf-8")
def pip_install(*args):
    # ставим пакеты и, если pip упал, показываем конец его сообщения, чтобы сразу было видно причину
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip не смог поставить пакеты, причина напечатана выше")

pip_install("-r", "requirements-podcast-colab.txt")
# silero-vad ставим без зависимостей: иначе pip может заменить torch из образа colab на другую сборку
pip_install("--no-deps", "silero-vad==6.2.3")
# отключаем лишние предупреждения токенизаторов и телеметрию hugging face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print("Пакеты установлены. CUDA-версия torch берётся из образа Colab.")

In [ ]:
# Модули проекта лежат в source/ рядом с ноутбуком. В Colab получаем эти же версии с GitHub.
import base64
import hashlib
import importlib
import json
import platform
import random
import requests

SOURCE_REVISIONS = {'podcast_core.py': '9657f3978971f9f806698c8dcad960c952333d82',
 'podcast_models.py': '6b259444dd36ec8b995909f6263f284fdedbb2a3',
 'podcast_experiment.py': '334655fff1dd7a3ca39a414442f747744631bd13',
 'test_core.py': 'c42a7b922bdc6939d6627c887e951d15f7769afe',
 'radiot_metadata.py': 'd11b79e2fed4559094bd15a93a574eadcbfa883a'}

RUNTIME = Path("podcast_runtime")
RUNTIME.mkdir(exist_ok=True)
SOURCE_FILES = {}
for name, revision in SOURCE_REVISIONS.items():
    local = Path("source") / name
    if local.is_file():
        data = local.read_bytes()
    else:
        url = f"https://api.github.com/repos/majlychko/Group_Homework/git/blobs/{revision}"
        response = requests.get(url, timeout=40)
        response.raise_for_status()
        data = base64.b64decode(response.json()["content"])
    actual = hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()
    if actual != revision:
        raise ValueError(f"Содержимое {name} отличается от версии, использованной для проекта")
    SOURCE_FILES[name] = data.decode("utf-8")
    (RUNTIME / name).write_bytes(data)
sys.path.insert(0, str(RUNTIME.resolve()))
from podcast_core import *
from podcast_experiment import Experiment
# общие библиотеки для таблиц, графиков и вывода
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, HTML, Audio, FileLink
from packaging.version import Version
# nemotron и qwen требуют свежий torch, проверяем это сразу
assert Version(torch.__version__.split("+")[0]) >= Version("2.6.0"), "Нужен torch >= 2.6: используйте свежую среду Colab"
# если в форме выбран gpu, а colab его не дал, останавливаемся с понятной ошибкой
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("GPU недоступен. В Colab выберите T4 GPU и запустите заново; CPU включается явно в форме.")
# фиксируем все генераторы случайных чисел, чтобы прогон повторялся
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# снимок метаданных шести выпусков радио-т на 19.11.2026: ссылки, разбиение dev/test и авторские главы
from radiot_metadata import load_manifest
# Разметку читаем с сайта при запуске. SHA-256 закрепляет версию всех названий и таймкодов.
PUBLIC_EPISODES = [{'id': 1029,
  'title': 'Радио-Т 1029',
  'page': 'https://radio-t.com/p/2026/09/05/podcast-1029/',
  'audio_url': 'https://cdn.radio-t.com/rt_podcast1029.mp3',
  'split': 'dev',
  'license': 'CC BY-NC-ND 3.0',
  'metadata_retrieved': '2026-11-19',
  'chapter_labels_sha256': '72122250db921a849093c356ec8b6b69fe41d309b3913db153f66bb4b9f4ace3'},
 {'id': 1030,
  'title': 'Радио-Т 1030',
  'page': 'https://radio-t.com/p/2026/09/12/podcast-1030/',
  'audio_url': 'https://cdn.radio-t.com/rt_podcast1030.mp3',
  'split': 'dev',
  'license': 'CC BY-NC-ND 3.0',
  'metadata_retrieved': '2026-11-19',
  'chapter_labels_sha256': '11e889bcb2c807381b697fa72260271fb469f6a05861441fb6dbae3565cd7a30'},
 {'id': 1031,
  'title': 'Радио-Т 1031',
  'page': 'https://radio-t.com/p/2026/09/19/podcast-1031/',
  'audio_url': 'https://cdn.radio-t.com/rt_podcast1031.mp3',
  'split': 'test',
  'license': 'CC BY-NC-ND 3.0',
  'metadata_retrieved': '2026-11-19',
  'chapter_labels_sha256': 'f5454dc8fbf44740f698e2232917e3f592d73aaf09acd1c949c47d3d816a6c52'},
 {'id': 1032,
  'title': 'Радио-Т 1032',
  'page': 'https://radio-t.com/p/2026/09/26/podcast-1032/',
  'audio_url': 'https://cdn.radio-t.com/rt_podcast1032.mp3',
  'split': 'test',
  'license': 'CC BY-NC-ND 3.0',
  'metadata_retrieved': '2026-11-19',
  'chapter_labels_sha256': '03316ae655d7561cce3cc75e0194f2e074447a74bbf945ced3554d47f8622b51'},
 {'id': 1033,
  'title': 'Радио-Т 1033',
  'page': 'https://radio-t.com/p/2026/10/03/podcast-1033/',
  'audio_url': 'https://cdn.radio-t.com/rt_podcast1033.mp3',
  'split': 'test',
  'license': 'CC BY-NC-ND 3.0',
  'metadata_retrieved': '2026-11-19',
  'chapter_labels_sha256': 'bb3fd3e5c815ab99c8f52d4b8ed23d93b6bd6e1ec624aee3b0b6a2aa0444c565'},
 {'id': 1034,
  'title': 'Радио-Т 1034',
  'page': 'https://radio-t.com/p/2026/10/17/podcast-1034/',
  'audio_url': 'https://cdn.radio-t.com/rt_podcast1034.mp3',
  'split': 'test',
  'license': 'CC BY-NC-ND 3.0',
  'metadata_retrieved': '2026-11-19',
  'chapter_labels_sha256': 'aa2b8c2eca01e04e1dae2c8dc431e3d9ae408223cc66be39427599ead57f348b'}]
MANIFEST = load_manifest(PUBLIC_EPISODES)
# закреплённые коммиты весов моделей на hugging face, чтобы через месяц скачались те же веса
MODEL_REVISIONS = [{'id': 'ai-sage/GigaAM-v3', 'sha': '7655ad717f8122257385bb4b2f373db3697e8680', 'branch': 'e2e_rnnt', 'license': 'mit', 'gated': False}, {'id': 'openai/whisper-large-v3-turbo', 'sha': '41f01f3fe87f28c78e2fbf8b568835947dd65ed9', 'license': 'mit', 'gated': False}, {'id': 'nvidia/parakeet-tdt-0.6b-v3', 'sha': '541d1f99c6b0c3cd0b11a95167540bb8edefd82b', 'license': 'cc-by-4.0', 'gated': False}, {'id': 'Qwen/Qwen3-ASR-1.7B-hf', 'sha': 'bcd2b5b7f32b480ab5790554cfa8347f246a14f3', 'license': 'apache-2.0', 'gated': False}, {'id': 'nvidia/Nemotron-3-Diarization', 'sha': '52a432672732425e4c19765b9fec9b5810e6f4ba', 'license': 'openmdw-1.1', 'gated': False}, {'id': 'Qwen/Qwen3-Embedding-0.6B', 'sha': '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3', 'license': 'apache-2.0', 'gated': False}, {'id': 'Qwen/Qwen3-4B-Instruct-2507', 'sha': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'license': 'apache-2.0', 'gated': False}, {'id': 'deepvk/USER2-base', 'sha': '94bbff0cc696f26dc93b275fc874657727df0c29', 'license': 'apache-2.0', 'gated': False}]
# папка запуска, туда сохраняем всё, что считаем
WORK = Path(WORK_DIRECTORY); WORK.mkdir(parents=True, exist_ok=True)
save_json(WORK / "model_revisions.json", MODEL_REVISIONS)
# записываем, на какой среде шёл прогон: версии python и torch, gpu, профиль и seed
environment = {"python": platform.python_version(), "torch": torch.__version__, "device": DEVICE,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               "profile": RUN_PROFILE, "seed": SEED}
save_json(WORK / "environment.json", environment)
# и полный список установленных пакетов
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
(WORK / "requirements-executed.txt").write_text(freeze, encoding="utf-8")
display(pd.Series(environment).to_frame("Среда запуска"))


In [ ]:
# прогоняем автотесты чистых функций (метрики, сегментация, экономика)
# искусственные примеры в тестах не попадают в test-выборку
import unittest
spec = importlib.util.spec_from_file_location("podcast_tests", RUNTIME / "test_core.py")
tests_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(tests_module)
test_result = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromModule(tests_module))
# если хоть одна проверка упала, дальше не идём
assert test_result.wasSuccessful(), "Автоматические проверки не прошли"

## 2. заказчик, аудитория и экономика процесса

заказчик у нас из брифа: продюсерская студия, которая делает 5-10 часовых видеоподкастов в неделю для нескольких брендов. живого интервью со студией мы не проводили, поэтому всё, что касается её процессов, дальше идёт как допущение. пользователь инструмента это продюсер или монтажёр, а решение о внедрении принимает руководитель производства.

что болит у продюсера: перед публикацией нужно переслушать выпуск, разбить его на главы, придумать названия и написать описание. наш инструмент делает черновик, где каждая глава привязана ко времени записи. главная бизнес-метрика для нас это минуты активной работы редактора на один часовой выпуск. время, пока GPU обрабатывает запись, мы считаем отдельно: человек в это время может заниматься другим.

чтобы прикинуть размер рынка, мы взяли данные РАЭК за 2023 год: там говорится о **более чем 50 профессиональных студиях подкастов** (https://www.raec.ru/live/branch/14160/). понятно, что это не текущий TAM и не список готовых клиентов, а скорее ориентир. продакшн-центры из той же статьи мы не прибавляли, потому что они могут пересекаться со студиями. для проверки экономики нам хватает одной студии, а сценарий на 50 студий ниже подписан как допущение.

все ставки в таблице ниже сценарные, мы их не выдаём за рыночную статистику. 1000 ₽ за час редактора уже включают налоги и взносы работодателя, сверху мы ничего не начисляем. 150 ₽ за час GPU тоже допущение для анализа чувствительности, а не цена конкретного облака.

In [ ]:
# берём допущения экономики из одного места (класс Economics в podcast_core)
ECON = Economics()
# показываем таблицу допущений, чтобы было видно, на чём держится расчёт
assumptions = pd.DataFrame([
    ["Выпусков в месяц", ECON.episodes_month, "Допущение: 7.5 * 52 / 12, внутри брифа 5-10 в неделю"],
    ["Минут вручную на часовой выпуск", ECON.manual_min, "Допущение: 50 на главы + 25 на описание, проверить хронометражем"],
    ["Полная стоимость часа редактора, ₽", ECON.loaded_hour_rub, "Допущение заказчика, без повторного начисления налогов"],
    ["Рабочих часов в FTE за месяц", ECON.work_hours_month, "Расчётная норма, допущение"],
    ["Внедрение, часов", ECON.setup_hours, "Сметное допущение: код, проверка, интерфейс, документация"],
    ["Стоимость часа разработки, ₽", ECON.developer_hour_rub, "Допущение"],
    ["Поддержка, часов в месяц", ECON.support_hours_month, "Допущение"],
    ["GPU, ₽ за вычислительный час", ECON.gpu_hour_rub, "Сценарий, цену поставщика перед внедрением запросить отдельно"],
], columns=["Параметр", "Значение", "Основание"])
display(assumptions)
assumptions.to_csv(WORK / "economic_assumptions.csv", index=False, encoding="utf-8-sig")
# считаем ёмкость процесса для 5, 7.5 и 10 выпусков в неделю: часы ручной работы, fte и деньги
capacity = []
for weekly in [5, 7.5, 10]:
    monthly = weekly * 52 / 12
    hours = monthly * ECON.manual_min / 60
    capacity.append({"выпусков в неделю": weekly, "выпусков в месяц": monthly,
                     "часов ручной работы": hours, "текущие FTE": hours / ECON.work_hours_month,
                     "стоимость процесса, ₽/мес": hours * ECON.loaded_hour_rub})
display(pd.DataFrame(capacity).round(2))
# базовый сценарий - середина диапазона, 7.5 выпуска в неделю
base_capacity = capacity[1]
display(Markdown(f"В базовом сценарии процесс занимает **{base_capacity['часов ручной работы']:.1f} ч/мес**, "
                 f"или **{base_capacity['текущие FTE']:.3f} FTE**. Это ёмкость процесса; высвобождение ещё не измерено."))

**Вывод по блоку.** При 7,5 часовых выпусках в неделю и 75 минутах ручной подготовки на выпуск получается 40,6 часа в месяц, или 0,254 ставки. При 10 выпусках в неделю нагрузка достигает 54,2 часа, или 0,34 ставки. Это исходные допущения для пилота.

## 3. как качество модели превращается в рабочее время

основная ML-метрика у нас F1 по внутренним границам глав с допуском ±30 секунд. рядом считаем precision и recall, F1@15, F1@60, а также Pk и WindowDiff на сетке 10 секунд. начало выпуска (00:00) и его конец за найденные границы не считаем. каждое предсказание может совпасть только с одной эталонной границей: пары подбираем динамическим программированием, сначала максимизируем число совпадений, потом минимизируем суммарный сдвиг. так метрика не завышается из-за того, что одну границу засчитали дважды.

сам по себе F1 не говорит, сколько времени уйдёт на правки, поэтому в экономику мы передаём TP, FP, FN и число предложенных глав по отдельности. важная мелочь: граница и глава это разные вещи, при B границах получается B+1 глава.

$$T_{after}=8+0.75\cdot N_{chapters}+0.5\cdot FP+10\cdot FN+1.5\cdot N_{chapters}\cdot(1-A).$$

все коэффициенты здесь в минутах и это допущения. A это доля названий, которые редактор принял без правок. пока ручной оценки нет, берём три сценария: A=0.25, 0.5 и 0.75. семантическое сходство или hit@1 мы вместо A не подставляем. в 8 минут фиксированной проверки входит просмотр описания и меток спикеров; если диаризацию придётся долго исправлять, этот коэффициент надо будет поднять.

пропущенная граница (FN) у нас стоит 10 минут, и это сделано специально. чтобы найти пропущенную смену темы, редактору нужно переслушать кусок записи, а это примерно столько же, сколько уходит на одну главу при ручной разметке (75 минут на 7 глав, около 10 минут на главу). Например, при семи эталонных и семи предложенных главах без совпавших внутренних границ получаем 6 FP и 6 FN. При A=0,5 прогноз составляет 81,5 минуты: 8 + 0,75 × 7 + 0,5 × 6 + 10 × 6 + 1,5 × 7 × 0,5. В разделе 10 сравниваем два черновика с разным числом и расположением границ.

если известны только P и R, то при G эталонных границах TP=R·G, FP=TP/P-TP, FN=G-TP. в самом эксперименте мы берём реальные счётчики, так что деления на нулевой precision там не бывает. время сверху мы не ограничиваем ручным процессом: плохой черновик тоже съедает время.

FTE и рубли здесь означают стоимость освободившегося времени. реальной экономией бюджета это станет, только если студия меньше заплатит подрядчикам, сократит переработки или не наймёт ещё одного человека. к сокращению штата модель не призывает.

In [ ]:
# смотрим, как качество границ переводится во время продюсера, ещё до запуска моделей
# для простоты берём p = r = f1, семь эталонных границ на выпуск и долю принятых названий 0.5
quality_scenarios = []
for f1 in [0.4, 0.5, 0.6, 0.7, 0.8]:
    r = economics_from_pr(f1, f1, n_gold=7, acceptance=0.5, config=ECON)
    quality_scenarios.append({"P=R=F1, допущение": f1, "мин после": r["minutes_after"],
                              "FTE высвобождается": r["fte_saved"], "чистый эффект, ₽/мес": r["net_month_rub"],
                              "окупаемость, мес": r["payback_months"], "ROI за 24 мес, %": 100*r["roi"]})
q = pd.DataFrame(quality_scenarios)
display(q.round(2))
# сколько минут на выпуск даёт рост качества с 0.6 до 0.7
change = q.iloc[3]["мин после"] - q.iloc[2]["мин после"]
display(Markdown(f"В этом сценарии рост P и R с 0.6 до 0.7 изменяет трудозатраты на **{change:+.2f} мин/выпуск**. "
                 "Это арифметика модели затрат, а не наблюдавшийся эффект внедрения."))

**Вывод по блоку.** В таблице принято семь эталонных **границ**, то есть восемь глав, и P = R. При F1 = 0,4 прогноз составляет 64,1 минуты на выпуск, при F1 = 0,7 — 42,1 минуты. На каждую пропущенную границу заложено 10 минут поиска, на удаление лишней — 0,5 минуты. Соотношение этих затрат предстоит проверить с редактором.

## 4. данные и выбор инструментов

данные у нас такие: шесть русскоязычных выпусков подкаста «Радио-Т», с №1029 по №1034. аудио и авторские главы лежат в открытом доступе (https://radio-t.com/podcast.rss и страницы выпусков). снимок метаданных зашит в ноутбук на 19.11.2026, так что если выпуск пропадёт с сайта, он не подменится другим. на dev идут 1029 и 1030, на test 1031-1034. быстрый профиль quick берёт по 60 минут из двух dev и двух test-выпусков, это проверка, что всё запускается, а не замена полного эксперимента.

авторские главы мы используем как эталон, но эталон слабый: это субъективный выбор ведущих, а не разметка всех смен темы. ни сегментатор, ни генератор эти названия не видят, их видит только оценщик. два dev-выпуска это немного для настройки, четыре test-выпуска немного для выводов. бутстреп мы делаем по выпускам, поэтому интервалы скорее описательные и перенос на другой жанр не доказывают.

как устроен пайплайн. аудио приводим к 16 кГц моно через ffmpeg. silero VAD находит отрезки речи, мы склеиваем их в куски до 20 секунд и передаём каждому из четырёх кандидатов ASR на dev, а на test — только выбранному Whisper. Время каждого куска известно из нарезки. дальше текст раскладываем по 30-секундным блокам по середине куска, сравниваем смысл слева и справа через косинусное сходство, и глубокие провалы связности становятся кандидатами в границы. сколько будет глав и какой минимальной длины, решают параметры, подобранные на dev. TF-IDF обучаем только на dev.

| этап | инструмент | почему он и какие ограничения |
|---|---|---|
| распознавание | четыре кандидата: GigaAM-v3 e2e-rnnt (Сбер, ноябрь 2025), MIT; Whisper large-v3-turbo (OpenAI, 2024), MIT; Parakeet-TDT-0.6B-v3 (NVIDIA, 2025), CC-BY-4.0; Qwen3-ASR-1.7B (Alibaba, 2026), Apache 2.0 | GigaAM проверяли на русском в ДЗ1, Whisper и Parakeet дают многоязычные варианты, а Qwen3-ASR представляет модели на основе LLM. какую брать, решаем на dev по качеству глав (раздел 5). всем даём одинаковые куски silero VAD до 20 секунд |
| диаризация | Nemotron-3-Diarization (NVIDIA, репозиторий создан в 2026 году), OpenMDW-1.1 | открытые веса без авторизации, потоковый speaker cache между чанками, до 8 спикеров, поддержка в transformers 5.23.0 |
| сегментация | USER2-base (deepvk, 2025), Apache-2.0; Qwen3-Embedding-0.6B (2025), Apache-2.0; TF-IDF | один алгоритм и одна сетка настройки на всех; победителя выбираем на dev. USER2 обучена на русском и держит контекст до 8192 токенов, Qwen крупнее и многоязычная, TF-IDF самый дешёвый |
| названия и суть | Qwen3-4B-Instruct-2507 (2025), Apache-2.0 | локальная генерация в fp16 на T4, без платного API. в промпт идёт выдержка из начала, середины и конца главы, поэтому детали могут теряться |
| рубрика | USER2-base и фиксированный список тем | zero-shot навигация; без ручных меток качество рубрик мы не измеряли |
| описание | шаблон из названий глав | второй раз LLM не вызываем, чтобы не добавлять новых фактов; текст всё равно проверяет редактор |

USER2-base вышла в 2025 году и обучалась именно на русском; в этом проекте она служит как кандидатом для сегментации, так и фиксированным оценщиком названий. ниже отдельная ячейка сама спрашивает у Hugging Face дату публикации и число лайков каждой модели.

меткам `speaker_0`, `speaker_1` мы не приписываем личности и роли, имена продюсер ставит руками. доли говорящих считаем по пересечению интервалов, при одновременной речи нормируем по сумме голосового времени.

карточки моделей: [GigaAM-v3](https://huggingface.co/ai-sage/GigaAM-v3), [Whisper large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo), [Parakeet-TDT-0.6B-v3](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3), [Qwen3-ASR-1.7B](https://huggingface.co/Qwen/Qwen3-ASR-1.7B-hf), [Nemotron](https://huggingface.co/nvidia/Nemotron-3-Diarization), [USER2-base](https://huggingface.co/deepvk/USER2-base), [Qwen Embedding](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), [Qwen Instruct](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). коммиты весов закреплены в `model_revisions.json`.

у «Радио-Т» лицензия CC BY-NC-ND 3.0 (https://radio-t.com/p/2026/09/05/podcast-1029/). поэтому аудио, расшифровки и тексты, которые мы из них сгенерировали, в публичный репозиторий не кладём. для коммерческого продукта нужны записи самого заказчика и права на них, наш учебный эксперимент таких прав не даёт.

### насколько свежие модели

чтобы не спорить на словах, спрашиваем у Hugging Face дату создания репозитория, число лайков и скачиваний для каждой модели пайплайна. если интернета нет, ячейка просто пропустится и ничего не сломает.

In [ ]:
# спрашиваем у hugging face дату создания, лайки и скачивания каждой модели пайплайна
from huggingface_hub import HfApi
rows = []
for item in MODEL_REVISIONS:
    try:
        # берём сведения о репозитории модели, это открытый запрос без токена
        info = HfApi().model_info(item["id"])
        rows.append({"модель": item["id"], "создана": str(info.created_at)[:10], "лайков": info.likes,
                     "скачиваний за месяц": info.downloads, "лицензия": item["license"]})
    except Exception as error:
        # без интернета просто пишем причину, остальной ноутбук от этого не зависит
        rows.append({"модель": item["id"], "создана": "нет ответа", "лайков": None,
                     "скачиваний за месяц": None, "лицензия": item["license"]})
model_table = pd.DataFrame(rows)
display(model_table)
model_table.to_csv(WORK / "model_freshness.csv", index=False)

In [ ]:
# создаём эксперимент: список выпусков, разбиение на dev и test, ревизии моделей
exp = Experiment(WORK, MANIFEST, MODEL_REVISIONS, DEVICE, EVAL_LIMIT_MIN, quick=RUN_PROFILE == "quick")
# скачиваем выпуски радио-т и переводим в 16 кгц моно
display(exp.prepare_audio().round(2))
# считаем, сколько часов аудио реально попало в этот запуск
audio_hours = exp.data_table.minutes.sum() / 60
display(Markdown(f"загружено **{len(exp.data_table)} выпусков**, **{audio_hours:.2f} ч аудио**; "
                 f"dev={exp.ids['dev']}, test={exp.ids['test']}. "
                 "Это фактический объём данного запуска, а не план выборки."))

**Вывод по блоку.** Обработали шесть выпусков одного шоу общей длительностью 14,65 часа. Снимок Hugging Face охватывает 2024–2026 годы: репозиторий Whisper large-v3-turbo создан в октябре 2024 года, GigaAM-v3 — в ноябре 2025-го, Qwen3-ASR — в июне 2026-го, Nemotron — в сентябре 2026-го. Эти даты относятся к репозиториям. Выбор модели для глав определяет результат на dev.

## 5. распознавание: четыре модели на dev, потом потоковая диаризация

распознавание мы не берём на веру: на dev прогоняем четыре открытые модели и выбираем ту, с которой главы получаются лучше.

| модель | почему кандидат | чего опасаемся |
|---|---|---|
| GigaAM-v3 e2e-rnnt (Сбер, 2025), MIT | сделана под русский, в ДЗ1 показала лучший WER, сама ставит пунктуацию | английские термины и названия продуктов |
| Whisper large-v3-turbo (OpenAI, 2024), MIT | хорошо знает английские слова внутри русской речи, язык можно зафиксировать | иногда дописывает фразы, которых не было (галлюцинации на паузах) |
| Parakeet-TDT-0.6B-v3 (NVIDIA, 2025), CC-BY-4.0 | очень быстрая, свободная лицензия для коммерции, пунктуация и таймкоды слов | язык определяет сама и может уйти в английский или украинский; незнакомые слова распознаёт плохо |
| Qwen3-ASR-1.7B (Alibaba, 2026), Apache 2.0 | самая свежая: распознавание на основе LLM, язык фиксируется, авторы заявляют лучший результат среди открытых моделей | цифр на русской разговорной речи нет; как у любой LLM, возможны повторы и пропуски на длинных кусках |

всем четырём даём одинаковый вход: silero VAD режет запись на куски до 20 секунд, дальнейшая обработка этих кусков зависит от модели. Parakeet целиком берёт не больше 24 минут записи, так что нарезка ей и так нужна.

ручной расшифровки этих подкастов у нас нет, поэтому WER относительно правды мы не считаем. вместо этого для каждой модели повторяем на dev весь подбор сегментации (три способа эмбеддингов на 24 варианта параметров) и берём лучший F1@30. так модель распознавания выбирается по тому, что важно продюсеру, то есть по главам, и тем же правилом, что и эмбеддер в разделе 6. рядом показываем скорость, число слов на час, расхождение текста с GigaAM (это не WER, а только мера того, насколько модели расходятся между собой) и долю кусков, которые пометила эвристика по буквам. Флаг срабатывает, если более половины букв латинские или встречаются і, ї, є, ґ. Такая проверка не определяет язык аудио. при равенстве F1 используется заранее заданный порядок кандидатов. test распознаём только выбранной моделью, в выборе он не участвует.

в ДЗ1 (Еремина) WER мерили на Golos и записях звонков; это другие данные, поэтому те цифры мы используем только как исходную гипотезу. DER без разметки «кто когда говорит» тоже не посчитать.

модели загружаем строго по очереди, чтобы уложиться в память T4: кандидат ASR, освобождаем, следующий кандидат, освобождаем, Nemotron, эмбеддер, LLM. в ключ кэша входят аудио и его SHA-256, коммит модели и параметры, так что при повторном запуске ничего не пересчитывается зря. По журналу timings.csv распознавание двух dev-выпусков четырьмя кандидатами заняло в сумме около 65,5 минуты вычислений. Qwen3-ASR подключаем через встроенную поддержку transformers 5, а не через пакет qwen-asr: тот требует transformers 4.57 и сломал бы остальной пайплайн.

In [ ]:
# распознаём dev четырьмя моделями, выбираем модель по f1@30 глав на dev и распознаём test только ею
asr_table = exp.transcribe()
# сравнение кандидатов на dev: качество глав, скорость, расхождение текстов и доля кусков не на русском
display(exp.asr_table.round(4))
first = exp.asr_names[0]
selected_row = exp.asr_table.set_index("asr").loc[exp.asr_selected]
display(Markdown(f"на dev лучше всего главы получились с **{exp.asr_selected}**: F1@30 {selected_row['dev_F1@30']:.3f} "
                 f"(метод {selected_row['лучший метод']}). "
                 + "; ".join(f"{r.asr}: F1@30 {r['dev_F1@30']:.3f}, RTF {r.rtf_dev:.3f}, по эвристике алфавита {r['доля кусков не на русском']:.1%}"
                             for _, r in exp.asr_table.iterrows())
                 + f". эта модель распознаёт test и используется в текущем пайплайне MVP."))
# итог распознавания по всем выпускам: число сегментов, слов и скорость
display(asr_table.round(4))

In [ ]:
# потоковая диаризация nemotron на test-выпусках: кто и когда говорит
diar_table = exp.diarize_test()
# сколько спикеров нашла модель и с какой скоростью
display(diar_table.round(4))

**Вывод по блоку.** На двух dev-выпусках связка Whisper large-v3-turbo и TF-IDF дала macro F1@30 = 0,417. Лучший вариант с каждым из трёх остальных ASR дал 0,333. Whisper затем распознал четыре test-выпуска. Его RTF на dev равен 0,062, на test — 0,063, или примерно 3,8 минуты вычислений на час аудио. Самый быстрый ASR на dev — Parakeet (RTF 0,031). На каждом test-выпуске Nemotron выделил четыре метки голоса при RTF около 0,061. Без ручной разметки неизвестны WER, DER и соответствие меток реальным людям.

## 6. настройка на dev и фиксация решения

перебираем ширину окна (2, 4 или 6 блоков), целевую длину главы (300, 600, 900 или 1200 секунд) и сглаживание (1 или 3). блок у нас 30 секунд, минимальная глава 90 секунд. главный критерий macro F1@30 по двум dev-выпускам, а при равенстве выигрывает более дешёвый метод. это обычный подбор параметров для готовых моделей, нейросети мы не обучаем.

для сравнения есть два простых baseline. «равномерный» режет запись на куски той длины, что выбрана на dev; сколько глав в test-выпуске, он не знает. «случайный» соблюдает минимальную длину главы и усредняется по 20 фиксированным seed. TF-IDF тоже по сути содержательный baseline, и на узкой выборке он вполне может победить.

In [ ]:
# считаем векторы 30-секундных блоков тремя способами: tf-idf, user2-base и qwen-embedding
exp.make_embeddings()
# перебираем параметры на dev и показываем лучший вариант каждого метода
display(exp.tune().round(4))
# с этого момента выбор зафиксирован, test его не меняет
print("Зафиксирован выбор по dev:", exp.selected, asdict(exp.configs[exp.selected]))
print("Целевая длина равномерного baseline:", exp.baseline_target, "секунд")

**Вывод по блоку.** На расшифровке Whisper лучшие результаты на dev составили 0,417 у TF-IDF, 0,294 у USER2-base и 0,203 у Qwen3-Embedding по macro F1@30. Для test зафиксировали TF-IDF: блок 30 секунд, окно 6 блоков, целевая длина главы 1200 секунд, минимум 90 секунд, сглаживание 1. Равномерный baseline настроили отдельно на dev, его шаг — 300 секунд.

## 7. test: сравнение, неопределённость и ошибки

с этого места test мы для настройки не трогаем. если по его результатам захочется поменять метод, test фактически станет вторым dev и понадобится новый. сравнение на test это отчёт по кандидатам, которые закрепили заранее, а в продукт идёт метод, выбранный на dev.

In [ ]:
# один раз оцениваем все методы и baseline на test
summary = exp.evaluate()
display(summary[["method", "selected_on_dev", "n_episodes", "F1@15", "F1@30", "F1@30_lo", "F1@30_hi", "F1@60", "P@30", "R@30", "Pk", "WindowDiff"]].round(3))
# парная разница выбранного метода с равномерной нарезкой и её бутстреп-интервал
delta, delta_lo, delta_hi = exp.paired_delta
# формулировку вывода выбираем по тому, что реально получилось
verdict = ("наблюдаемый прирост положителен, но интервал включает ноль" if delta > 0 and delta_lo <= 0
           else "в этом запуске преимущество не получено" if delta <= 0
           else "парный bootstrap-интервал прироста положителен на этой небольшой выборке")
display(Markdown(f"выбран `{exp.selected}`. Парная разница F1@30 с равномерной нарезкой: "
                 f"**{delta:+.3f}**, 95% bootstrap **[{delta_lo:+.3f}; {delta_hi:+.3f}]**; {verdict}. "
                 "О переносе на брендовые интервью по этому эксперименту судить рано."))

In [ ]:
# рисуем f1@30 всех методов на test с бутстреп-интервалами; выбранный метод выделяем цветом
fig, ax = plt.subplots(figsize=(8, 4))
plot = summary.set_index("method")
positions = np.arange(len(plot))
ax.bar(positions, plot["F1@30"], color=["#1856a7" if n == exp.selected else "#a8b5c8" for n in plot.index])
# усы - это 95% интервал по выпускам
ax.errorbar(positions, plot["F1@30"], yerr=np.maximum(0, np.array([plot["F1@30"]-plot["F1@30_lo"], plot["F1@30_hi"]-plot["F1@30"]])), fmt="none", ecolor="#243244", capsize=4)
ax.set(xticks=positions, xticklabels=plot.index, ylabel="Macro F1@30", ylim=(0, 1), title="Сравнение границ с авторскими главами на test")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout(); fig.savefig(WORK / "test_f1.png", dpi=160); plt.show()
# подпись зависит от знака разницы, чтобы не написать «превосходит», если метод проиграл
comparison = "превосходит" if delta > 0 else "не превосходит"
display(Markdown(f"выбранный на dev метод {comparison} uniform по среднему F1@30: разница **{delta:+.3f}**. "
                 "Интервалы получены пересэмплированием выпусков; случайный baseline сначала усреднён по 20 нарезкам каждого выпуска."))

In [ ]:
# разбираем ошибки на самом трудном test-выпуске
selected_rows = exp.per_episode[exp.per_episode.method == exp.selected].sort_values("F1@30")
display(selected_rows[["episode", "n_pred", "n_gold", "TP@30", "FP@30", "FN@30", "F1@30", "Pk"]].round(3))
# худший выпуск по f1@30
worst_id = int(selected_rows.iloc[0].episode)
ep = exp.episodes[worst_id]
pred = exp.predictions[(worst_id, exp.selected)]
# находим, какие границы совпали, а какие нет
matched = match_boundaries(pred["bounds"], ep["gold_bounds"], 30)
used_p, used_r = {x[0] for x in matched}, {x[1] for x in matched}
# для каждой лишней и пропущенной границы берём текст ±45 секунд вокруг неё, чтобы потом прослушать и объяснить
errors = []
for kind, times, used in [("FP: лишняя граница", pred["bounds"], used_p), ("FN: пропущенная граница", ep["gold_bounds"], used_r)]:
    for j, t in enumerate(times):
        if j not in used:
            clip = chapter_segments(ep["transcript"]["segments"], max(0, t-45), min(ep["duration"], t+45))
            errors.append({"episode": worst_id, "error": kind, "time": timestamp(t), "seconds": t, "context": excerpt(clip, 90)})
error_table = pd.DataFrame(errors)
error_table.to_csv(WORK / "error_review.csv", index=False, encoding="utf-8-sig")
display(error_table.head(8))
# график глубины провалов: зелёные линии - авторские главы, красный пунктир - наши
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(np.array(pred["times"])/60, pred["depth"], label="Глубина смены связности")
for j, t in enumerate(ep["gold_bounds"]): ax.axvline(t/60, color="#338451", alpha=.6, label="Авторская граница" if j == 0 else None)
for j, t in enumerate(pred["bounds"]): ax.axvline(t/60, color="#bb4d31", ls="--", alpha=.7, label="Предсказание" if j == 0 else None)
ax.set(xlabel="Минуты выпуска", ylabel="Depth score", title=f"Где расходятся границы: выпуск {worst_id}"); ax.legend()
fig.tight_layout(); fig.savefig(WORK / "boundary_errors.png", dpi=160); plt.show()
worst = selected_rows.iloc[0]
display(Markdown(f"В выпуске **{worst_id}**: **{int(worst['FP@30'])} FP**, **{int(worst['FN@30'])} FN**, "
                 f"F1@30 = **{worst['F1@30']:.3f}**. Контексты выше нужны для ручного объяснения причин. "
                 "Растянутое вступление, плавный переход, реклама или ошибка ASR - это гипотезы, пока фрагмент не прослушан."))

**Вывод по блоку.** На четырёх test-выпусках выбранный TF-IDF получил macro F1@30 = 0,036, precision = 0,042 и recall = 0,031. Равномерная нарезка получила 0,113, случайная — 0,087. Парная разница TF-IDF с равномерной нарезкой составила −0,078, описательный 95%-й бутстреп-интервал по выпускам — от −0,144 до −0,012. У TF-IDF меньше Pk и WindowDiff (0,447 и 0,470 против 0,552 и 0,817), но по основной метрике он уступил.

Из 23 предложенных внутренних границ одна попала в допуск, 22 лишние и 30 пропущены. Единственное совпадение из 31 эталонной границы было в выпуске 1033. В 1031, 1032 и 1034 совпадений нет. График показывает один из этих трёх выпусков — 1031. USER2-base дал 0,189, Qwen3-Embedding — 0,255, лучший результат в сравнении на test. Чтобы выбрать Qwen по этому наблюдению, потребуется новая независимая проверка.

## 8. названия, рубрики и описание

LLM видит только текст главы, авторских названий у неё нет. она возвращает JSON с названием, короткой сутью и цитатой-опорой. Для проверки опоры требуется не менее трёх слов, из которых не менее 85% встречаются где-либо в тексте главы. Порядок и соседство слов не проверяются. Поэтому прошедшая проверку фраза может отличаться от дословной цитаты, а аннотация всё равно требует сверки по смыслу. если JSON сломан или не выполнен порог совпадения слов, ставим извлечённые ключевые слова с меткой `extractive_after_invalid_llm` и отдельно показываем, как часто так бывает.

названия сравниваем с двумя простыми вариантами («первые семь слов» и TF-IDF) на одних и тех же **эталонных интервалах**, чтобы ошибка границы не смешивалась с качеством названия. главы, найденные моделью, оцениваем отдельно. метрики тут косинус к названию автора и hit@1 среди глав выпуска, оценщик всегда USER2-base. это автоматические прокси, и называть их точностью одобрения редактором нельзя.

In [ ]:
# генерируем названия и суть глав через qwen3-4b
exp.generate()
# оцениваем названия автоматически и присваиваем рубрики
title_summary = exp.evaluate_titles_and_topics()
display(title_summary.round(3))
# сохраняем описания, главы и vtt для каждого test-выпуска
exp.export()
# считаем, как часто llm отвечала не по формату и мы ставили запасное название
production_titles = pd.DataFrame([c for (i, mode), chapters in exp.generations.items() if mode == "predicted" for c in chapters])
fallback_rate = (production_titles.generation_status == "extractive_after_invalid_llm").mean()
display(Markdown(f"В найденных главах **{len(production_titles)} названий**, доля перехода к извлекаемому заголовку "
                 f"после отказа LLM: **{fallback_rate:.1%}**. Это проверка формата и цитаты, а не оценка достоверности всех фактов."))
# шаблон для ручной оценки редакторами
print("Шаблон ручной оценки:", WORK / "human_review_template.csv")
display(exp.review[["episode", "chapter", "title", "summary", "evidence", "generation_status"]].head(8))

долю названий A, которые редактор принимает без правок, нужно мерить руками: два редактора независимо слушают главу и не видят авторского названия. пока таких оценок нет, в экономике A задаём сценариями 0.25, 0.5 и 0.75, а качество рубрик и DER остаются неизмеренными. автоматический hit@1 эти дыры не закрывает.

**Вывод по блоку.** На 35 авторских интервалах Qwen3-4B и заголовки TF-IDF дали одинаковый hit@1: 16 из 35 (0,457). У первых семи слов — 6 из 35 (0,171). Средний косинус к авторскому названию составил соответственно 0,720, 0,680 и 0,630. На 27 главах, найденных сегментатором, у генератора hit@1 = 11/27 (0,407), косинус = 0,688. Все 27 ответов прошли программную проверку формата и цитаты, запасные названия не потребовались. Редакторских оценок нет ни для одной из 27 глав, поэтому принятие названий и достоверность описаний ещё предстоит измерить.

## 9. дополнительное исследование: устойчивость к ошибкам в тексте

в расшифровку выбранного на dev Whisper мы специально добавляем удаления и замены слов (воспроизводимо, с фиксированным seed), настройки сегментатора не меняем. «добавленный WER» показывает, насколько испорченная копия отличается от исходной расшифровки. это не WER модели относительно настоящей речи. мы просто проверяем, насколько главы чувствительны к таким ошибкам: если распознавание на другой студии окажется хуже, сломается ли всё сразу.

In [ ]:
# стресс-тест: портим текст asr и смотрим, насколько падает качество глав
if RUN_STRESS_TEST:
    stress = exp.stress_test()
    display(stress.round(3))
    # график зависимости f1@30 от добавленного искусственного wer
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(100*stress.added_synthetic_WER, stress["F1@30"], marker="o", color="#1856a7")
    ax.set(xlabel="Добавленный синтетический WER, %", ylabel="Macro F1@30", ylim=(0, 1), title="Чувствительность к удалениям и заменам слов")
    fig.tight_layout(); fig.savefig(WORK / "synthetic_stress.png", dpi=160); plt.show()
    # насколько изменился f1 между чистым текстом и самым сильным искажением
    stress_change = stress.iloc[-1]["F1@30"] - stress.iloc[0]["F1@30"]
    display(Markdown(f"при максимальном искажении средний F1@30 изменился на **{stress_change:+.3f}**. "
                     "Направление и величина относятся к этой искусственной процедуре, выбранному методу и этим выпускам."))
else:
    print("Стресс-тест отключён. Он не влияет на основную оценку test.")

**Вывод по блоку.** Без искажения macro F1@30 выбранного TF-IDF равен 0,036. При фактически добавленном синтетическом WER около 10,0% получилось 0,113, а при 30,3% — 0,042. Изменение немонотонно. На четырёх выпусках с фиксированными случайными заменами и удалениями нельзя установить, как реальное ухудшение звука повлияет на сегментацию.

## 10. от измеренного качества к прогнозу экономии

в формулу идут реальные TP, FP и FN выбранного на dev метода, пересчитанные на час аудио. число глав учитывает, что каждый выпуск начинается с первой главы. коэффициенты правок, стоимость часа и принятие названий пока остаются допущениями, поэтому всё ниже это прогноз до пилота у заказчика.

RTF складываем только из тех этапов, которые будут работать в продакшене: распознавание, диаризация, выбранный эмбеддер, сегментация, генерация для найденных глав и рубрики. подбор параметров, проигравшие кандидаты, стресс-тест и оценка названий на эталонных интервалах в эксплуатационные затраты не входят. загрузку моделей добавляем как один пакетный запуск в месяц. скачивание выпусков и время людей за работу GPU не выдаём.

In [ ]:
# проверяем, загружены ли ручные оценки редакторов; если да, берём из них долю принятых названий a
title_acceptance = None
ratings_path = WORK / "human_review.csv"
if ratings_path.exists():
    ratings = pd.read_csv(ratings_path)
    # сначала проверяем, что файл заполнен по шаблону
    required = {"episode", "chapter", "title", "rater", "title_accepted"}
    assert required.issubset(ratings.columns), "У ручной оценки не хватает колонок шаблона"
    valid = ratings.dropna(subset=["rater", "title_accepted"]).copy()
    valid["title_accepted"] = pd.to_numeric(valid.title_accepted, errors="raise")
    assert valid.title_accepted.isin([0, 1]).all(), "Оценки должны быть 0 или 1"
    assert not valid.duplicated(["episode", "chapter", "rater"]).any(), "Повторная оценка одного редактора"
    # оценки должны относиться именно к главам этого прогона
    joined = valid.merge(exp.review[["episode", "chapter", "title"]], on=["episode", "chapter", "title"], how="inner")
    assert len(joined) == len(valid), "Файл оценок относится к другой версии глав"
    # нам надо, чтобы каждую главу посмотрели минимум два редактора
    counts = joined.groupby(["episode", "chapter"]).rater.nunique()
    covered = int((counts >= 2).sum())
    print(f"Глав с оценкой минимум двух редакторов: {covered}/{len(exp.review)}")
    if covered == len(exp.review):
        # если редакторы разошлись, считаем, что название требует правки
        title_acceptance = float(joined.groupby(["episode", "chapter"]).title_accepted.min().mean())
        print("Измеренное консервативное принятие названий:", title_acceptance)
    else:
        print("Для экономики остаются сценарии: полной независимой оценки ещё нет.")
# скорость продукта: rtf только по тем этапам, которые будут работать у заказчика
rtf, stages = exp.production_rtf()
display(stages.to_frame("RTF по production-этапам").round(4))
# подставляем измеренные ошибки в экономическую модель
display(exp.evaluate_economics(ECON, title_acceptance).round(3))
result = exp.economics_result
display(pd.Series({"статус": "прогноз по допущениям", "минут до": ECON.manual_min,
                   "минут после": result["minutes_after"], "часов до в месяц": result["hours_before"],
                   "часов после в месяц": result["hours_after"], "FTE до": result["fte_before"],
                   "FTE после": result["fte_after"], "FTE высвобождается": result["fte_saved"],
                   "эффект после эксплуатации, ₽/мес": result["net_month_rub"], "внедрение, ₽": result["capex_rub"],
                   "окупаемость, мес": result["payback_months"], "ROI за 24 мес, %": 100*result["roi"]}).to_frame("Значение"))
display(Markdown(f"на test измерен тёплый RTF **{rtf:.3f}**, то есть **{60*rtf:.1f} мин вычислений на час аудио**. "
                 f"При A={result['title_acceptance']:.2f} и остальных допущениях прогноз: **{result['minutes_after']:.1f} мин работы на выпуск**, "
                 f"**{result['fte_saved']:.3f} FTE** высвобождения. Фактическая экономия студии ещё не измерена."))
# для сопоставления сценариев применяем те же коэффициенты затрат к равномерной нарезке:
# она тоже даёт расшифровку и шаблон описания, но границы ставит вслепую, без понимания текста
uniform_rows = exp.per_episode[exp.per_episode.method == "uniform"]
uniform_hours = uniform_rows.duration.sum() / 3600
uniform_inputs = {"tp": uniform_rows["TP@30"].sum() / uniform_hours, "fp": uniform_rows["FP@30"].sum() / uniform_hours,
                  "fn": uniform_rows["FN@30"].sum() / uniform_hours, "rtf": exp.economics_inputs["rtf"],
                  "chapters_pred": (uniform_rows.n_pred.sum() + len(uniform_rows)) / uniform_hours}
uniform_result = economics_from_counts(**uniform_inputs, acceptance=result["title_acceptance"], config=ECON)
model_gain = uniform_result["minutes_after"] - result["minutes_after"]
display(Markdown(f"При тех же ставках равномерная нарезка даёт **{uniform_result['minutes_after']:.1f} мин** на часовой выпуск. "
                 f"Разница «равномерная минус выбранная» составляет **{model_gain:+.1f} мин** на выпуск "
                 f"и **{(uniform_result['hours_after'] - result['hours_after']) / ECON.work_hours_month:+.3f} FTE**. "
                 "На неё влияют число предложенных глав и цены ошибок. Реальные трудозатраты должен измерить редактор."))

In [ ]:
# чувствительность: меняем по одному допущению и смотрим, как двигаются минуты, fte и roi
from dataclasses import replace
sensitivity = []
for parameter, values in {"manual_min": [50, 75, 100], "episodes_month": [5*52/12, 7.5*52/12, 10*52/12],
                          "loaded_hour_rub": [600, 1000, 1500], "find_fn_min": [5, 10, 15],
                          "setup_hours": [40, 80, 160], "gpu_hour_rub": [50, 150, 300]}.items():
    for value in values:
        # копия допущений с одним изменённым параметром
        config = replace(ECON, **{parameter: value})
        r = economics_from_counts(**exp.economics_inputs, acceptance=result["title_acceptance"], config=config)
        sensitivity.append({"параметр": parameter, "значение": value, "мин после": r["minutes_after"],
                            "чистый эффект, ₽/мес": r["net_month_rub"], "FTE высвобождается": r["fte_saved"],
                            "окупаемость, мес": r["payback_months"], "ROI, %": 100*r["roi"]})
sensitivity_table = pd.DataFrame(sensitivity)
display(sensitivity_table.round(2)); sensitivity_table.to_csv(WORK / "economic_sensitivity.csv", index=False)
# сценарий масштаба: 50 похожих студий (оценка раэк по числу профессиональных студий)
market_process = 50 * ECON.episodes_month * ECON.manual_min / 60 * 12 * ECON.loaded_hour_rub
display(Markdown(f"сценарий **50 похожих студий**: стоимость ручного процесса **{market_process/1e6:.1f} млн ₽/год**. "
                 "Это модель затрат потенциального сегмента при одинаковом выпуске, а не измеренный рынок и не выручка сервиса. "
                 "Затраты на продажи, подключение и поддержку каждой студии, хранение и привлечение клиентов считаем в следующей ячейке."))

### масштабирование: сколько стоит продавать сервис многим студиям

выше мы считали одну студию, которая внедряет инструмент у себя. если продавать его как сервис многим студиям, появляются новые затраты: привлечение клиента (звонки, демо, пилот), подключение и обучение продюсера, поддержка каждой студии, хранение записей и доработка платформы под много клиентов. кроме того, часть студий будет уходить, и их место придётся покупать заново.

за основу берём экономию одной студии из нашего прогона, а цену подписки ставим в 30% от этой экономии, чтобы студии было выгодно платить. все ставки ниже это допущения, их надо проверить на пилоте и первых продажах.

In [ ]:
# считаем, во что обойдётся масштабирование: продажи, подключение, поддержка каждой студии, хранение и вычисления
# все ставки ниже это сценарные допущения, их надо уточнить на пилоте; качество модели берём из нашего прогона
SCALE = {
    "price_share": 0.30,          # цена подписки как доля сэкономленных студией денег, остальное остаётся студии
    "cac_rub": 30000.0,           # привлечение одной студии: около 20 часов менеджера на звонки, демо и пилот по 1500 ₽
    "onboarding_hours": 4.0,      # подключение студии: настройка, обучение продюсера, перенос шаблонов
    "support_hours_studio": 1.0,  # поддержка каждой студии в месяц сверх базовой поддержки сервиса
    "storage_gb_episode": 0.15,   # аудио часового выпуска в mp3 и результаты обработки
    "storage_months": 3,          # сколько месяцев храним записи студии
    "storage_rub_gb": 2.5,        # сценарная цена гигабайта облачного хранилища в месяц
    "platform_hours": 160.0,      # доработка под много клиентов: личные кабинеты, очередь задач, биллинг
    "churn_month": 0.03,          # доля студий, которые уходят каждый месяц; их место надо заново купить через CAC
}
gross_studio = result["gross_month_rub"]                      # сколько денег в месяц экономит одна студия
price = SCALE["price_share"] * gross_studio                   # подписка для студии
storage = ECON.episodes_month * SCALE["storage_months"] * SCALE["storage_gb_episode"] * SCALE["storage_rub_gb"]
variable = result["compute_month_rub"] + storage + SCALE["support_hours_studio"] * ECON.developer_hour_rub
margin = price - variable                                     # маржинальная прибыль сервиса с одной студии в месяц
onboarding = SCALE["onboarding_hours"] * ECON.developer_hour_rub
cac_payback = (SCALE["cac_rub"] + onboarding) / margin if margin > 0 else float("inf")
rub = lambda x: f"{x:,.0f}".replace(",", " ")  # пробел как разделитель тысяч
display(Markdown(f"юнит-экономика одной студии в месяц: подписка **{rub(price)} ₽** ({SCALE['price_share']:.0%} от её экономии {rub(gross_studio)} ₽), "
                 f"вычисления {rub(result['compute_month_rub'])} ₽, хранение {rub(storage)} ₽, поддержка "
                 f"{rub(SCALE['support_hours_studio'] * ECON.developer_hour_rub)} ₽, маржа **{rub(margin)} ₽**. "
                 f"привлечение и подключение стоят {rub(SCALE['cac_rub'] + onboarding)} ₽ и окупаются за **{cac_payback:.1f} мес**."))
rows = []
# одна студия у себя считается выше (внедрение без продаж), здесь сценарий сервиса для многих студий
for n in [10, 25, 50]:
    h = ECON.horizon_months
    # разовые затраты: базовое внедрение, доработка платформы под много клиентов, привлечение и подключение
    capex = ECON.setup_hours * ECON.developer_hour_rub + SCALE["platform_hours"] * ECON.developer_hour_rub
    acquisition = n * (SCALE["cac_rub"] + onboarding)
    # ушедших студий заменяем новыми, это повторное привлечение каждый месяц
    replacement = n * SCALE["churn_month"] * (SCALE["cac_rub"] + onboarding)
    monthly = n * margin - ECON.support_hours_month * ECON.developer_hour_rub - replacement
    invest = capex + acquisition
    rows.append({"студий": n, "выручка, ₽/мес": n * price, "переменные затраты, ₽/мес": n * variable,
                 "замена ушедших, ₽/мес": replacement, "операционная прибыль, ₽/мес": monthly,
                 "разовые вложения, ₽": invest, "окупаемость, мес": invest / monthly if monthly > 0 else float("inf"),
                 f"ROI за {h} мес, %": 100 * (h * monthly - invest) / invest,
                 "экономия студий за вычетом подписки, ₽/мес": n * (gross_studio - price)})
scale_table = pd.DataFrame(rows)
display(scale_table.round(1))
scale_table.to_csv(WORK / "scale_economics.csv", index=False, encoding="utf-8-sig")

### внутренняя экономика компании: почему подписка стоит столько и куда уходят деньги

юнит-экономика выше учитывает привлечение, подключение, поддержку, хранение и GPU. для честного расчёта этого мало: компания ещё платит налог с выручки, комиссию банку за приём оплаты, держит человека на сопровождении продукта, сервер, бухгалтерию и юриста, а GPU на практике работает дольше чистого времени обработки. кроме того, вложения и первое привлечение клиентов надо вернуть, а студии приходят не сразу, поэтому на старте нужны деньги.

ниже считаем по шагам: коридор цены (расчётные ориентиры стоимости внедрения и обслуживания студии), выбранную цену и тариф, отчёт о прибылях и убытках при 10, 25 и 50 студиях, показатели клиента (LTV, CAC, окупаемость привлечения), точки безубыточности и кассовый разрыв при подключении трёх студий в месяц. качество и время вычислений взяты из прогона, экономия студии рассчитана по сценарным ставкам, остальные ставки это допущения компании.

In [ ]:
# внутренняя экономика компании: из чего складывается цена подписки и сколько нам на самом деле стоит каждая студия
# здесь добавляем то, чего не было в юнит-экономике выше: налог, эквайринг, резерв gpu, постоянную команду,
# сервер, бухгалтерию и юриста, возврат вложений и кассовый разрыв на старте
# все ставки это сценарные допущения компании, их уточняем на первых продажах
INTERNAL = {
    "tax_share": 0.06,          # сценарное допущение: налог 6% с выручки, НДС в этой модели не учитывается
    "acquiring_share": 0.025,   # комиссия банка за приём оплаты картой или по счёту
    "gpu_reserve": 0.5,         # резерв к чистому времени gpu: холодный старт моделей, повторы упавших задач, простой
    "dev_hours_month": 16.0,    # сопровождение продукта: обновление моделей и библиотек, исправления, мониторинг
    "infra_rub_month": 5000.0,  # сервер для веб-интерфейса и очереди задач, резервные копии, мониторинг
    "backoffice_rub_month": 8000.0,  # бухгалтерия и юрист на аутсорсе
    "legal_setup_rub": 40000.0, # оферта, политика обработки персональных данных (голоса гостей это пдн), договоры
    "ramp_per_month": 3,        # сколько новых студий в месяц реально подключает один менеджер
    "price_episode_rub": 230.0, # решение по цене: рубли за обработанный выпуск, обоснование ниже
}
h = ECON.horizon_months
episodes = ECON.episodes_month
acq_one = SCALE["cac_rub"] + onboarding                       # привлечь и подключить одну студию
fixed_month = (INTERNAL["dev_hours_month"] * ECON.developer_hour_rub + INTERNAL["infra_rub_month"]
               + INTERNAL["backoffice_rub_month"])              # постоянные затраты компании в месяц
invest = (ECON.setup_hours + SCALE["platform_hours"]) * ECON.developer_hour_rub + INTERNAL["legal_setup_rub"]
variable_base = (result["compute_month_rub"] * (1 + INTERNAL["gpu_reserve"]) + storage
                 + SCALE["support_hours_studio"] * ECON.developer_hour_rub)  # gpu, хранение и поддержка одной студии
keep = 1 - INTERNAL["tax_share"] - INTERNAL["acquiring_share"]  # какая доля оплаты доходит до компании

def unit_margin(p):
    # сколько одна студия приносит в месяц после налога, эквайринга, переменных затрат и замены ушедших
    return p * keep - variable_base - SCALE["churn_month"] * acq_one

def required_price(n):
    # минимальная цена, при которой n студий за горизонт покрывают постоянные затраты и возвращают вложения
    return (variable_base + SCALE["churn_month"] * acq_one + acq_one / h + (fixed_month + invest / h) / n) / keep

# 1. коридор цены: потолок задаёт выгода студии, пол задаёт наша полная себестоимость
in_house_month = result["opex_month_rub"] + result["capex_rub"] / h  # студия делает сама: поддержка, gpu, внедрение на 24 мес
corridor = pd.DataFrame([
    {"студий": n, "нужная цена, ₽/мес": required_price(n), "₽ за выпуск": required_price(n) / episodes,
     "доля экономии студии, %": 100 * required_price(n) / gross_studio,
     "дешевле, чем делать самой": required_price(n) < in_house_month} for n in [10, 25, 50]])
display(corridor.round(1))
display(Markdown(
    f"Стоимость высвобожденного времени студии в сценарии — **{rub(gross_studio)} ₽/мес**. "
    f"Самостоятельное внедрение с распределением вложений на {h} мес. обойдётся в **{rub(in_house_month)} ₽/мес**. "
    f"Для сравнения, цена в упрощённом сценарии — {rub(price)} ₽/мес. "
    f"В расширенной модели для покрытия затрат и возврата вложений за {h} мес. нужны "
    f"{rub(required_price(10))} ₽/мес. при 10 студиях и {rub(required_price(50))} ₽/мес. при 50."))

# 2. решение: берём цену за выпуск с запасом над себестоимостью при 50 студиях и ниже альтернативы «сделать самой»
price_internal = INTERNAL["price_episode_rub"] * episodes
if not required_price(50) < price_internal < in_house_month:
    # на других данных цена может выйти из коридора, тогда её надо пересмотреть, а не скрывать
    print("внимание: выбранная цена вне коридора, пересмотрите price_episode_rub")
tariff = pd.DataFrame([{"выпусков в неделю": w, "выпусков в месяц": w * 52 / 12,
                        "подписка, ₽/мес": INTERNAL["price_episode_rub"] * w * 52 / 12,
                        "экономия студии, ₽/мес": gross_studio / episodes * w * 52 / 12,
                        "студии остаётся, ₽/мес": (gross_studio / episodes - INTERNAL["price_episode_rub"]) * w * 52 / 12}
                       for w in [5, 7.5, 10]])
display(tariff.round(1))
display(Markdown(
    f"цена **{INTERNAL['price_episode_rub']:.0f} ₽ за выпуск**, в базовом сценарии **{rub(price_internal)} ₽/мес** "
    f"({100 * price_internal / gross_studio:.0f}% экономии студии). она на {100 * (price_internal / required_price(50) - 1):.0f}% выше "
    f"полной себестоимости при 50 студиях и на {rub(in_house_month - price_internal)} ₽ ниже, чем студии стоит делать самой. "
    f"за {h} мес студия с подпиской сохраняет {rub(h * (gross_studio - price_internal))} ₽, "
    f"а при внедрении у себя {rub(h * result['net_month_rub'] - result['capex_rub'])} ₽ и рискует 120 тыс. ₽ заранее. "
    "Отношение тарифа к прогнозной экономии постоянно при заданной линейной зависимости от числа выпусков."))

# 3. отчёт о прибылях и убытках компании в месяц при устойчивом числе клиентов
def pnl(n, p=price_internal):
    revenue = n * p
    contribution = n * (p * keep - variable_base)
    ebitda = contribution - n * SCALE["churn_month"] * acq_one - fixed_month
    back = (invest + n * acq_one) / h                         # вложения и первое привлечение возвращаем за горизонт
    return {"выручка": revenue, "налог УСН 6%": -revenue * INTERNAL["tax_share"],
            "эквайринг": -revenue * INTERNAL["acquiring_share"],
            "GPU с резервом": -n * result["compute_month_rub"] * (1 + INTERNAL["gpu_reserve"]),
            "хранение": -n * storage, "поддержка студий": -n * SCALE["support_hours_studio"] * ECON.developer_hour_rub,
            "маржинальная прибыль": contribution, "замена ушедших студий": -n * SCALE["churn_month"] * acq_one,
            "сопровождение продукта": -INTERNAL["dev_hours_month"] * ECON.developer_hour_rub,
            "сервер и мониторинг": -INTERNAL["infra_rub_month"], "бухгалтерия и юрист": -INTERNAL["backoffice_rub_month"],
            "операционная прибыль": ebitda, f"возврат вложений за {h} мес": -back, "чистый результат": ebitda - back}
pnl_table = pd.DataFrame({f"{n} студий": pnl(n) for n in [10, 25, 50]})
display(pnl_table.round(0))
pnl_table.to_csv(WORK / "internal_pnl.csv", encoding="utf-8-sig")

# 4. показатели одного клиента и точки безубыточности при выбранной цене
import math

def minimum_studios(threshold):
    """Представление целого порога, в том числе для недостижимой безубыточности."""
    return str(math.ceil(threshold)) if math.isfinite(threshold) else "недостижимо"

margin_internal = price_internal * keep - variable_base
ltv = margin_internal / SCALE["churn_month"]                  # маржа за среднее время жизни клиента
# если студия не окупает даже замену ушедших, безубыточности нет ни при каком числе клиентов
breakeven_ops = fixed_month / unit_margin(price_internal) if unit_margin(price_internal) > 0 else float("inf")
full_unit = unit_margin(price_internal) - acq_one / h
breakeven_full = (fixed_month + invest / h) / full_unit if full_unit > 0 else float("inf")
unit_table = pd.Series({"маржа одной студии, ₽/мес": margin_internal,
                        "маржинальность, %": 100 * margin_internal / price_internal,
                        "среднее время жизни клиента, мес": 1 / SCALE["churn_month"],
                        "LTV, ₽": ltv, "CAC с подключением, ₽": acq_one, "LTV / CAC": ltv / acq_one,
                        "окупаемость привлечения, мес": acq_one / margin_internal,
                        "студий до безубыточности по операциям": breakeven_ops,
                        f"студий, чтобы вернуть вложения за {h} мес": breakeven_full}).to_frame("значение")
display(unit_table.round(2))

# 5. кассовый разрыв: студии приходят постепенно, а вложения и привлечение оплачиваются сразу
def cash_curve(target, months=36, p=price_internal):
    cash, clients, curve = -invest, 0, []
    for month in range(1, months + 1):
        new = min(INTERNAL["ramp_per_month"], target - clients)   # новые студии этого месяца
        clients += new
        cash += clients * unit_margin(p) - fixed_month - new * acq_one
        curve.append({"месяц": month, "студий": clients, "деньги нарастающим итогом, ₽": cash})
    return pd.DataFrame(curve)

fig, ax = plt.subplots(figsize=(8, 3.8))
cash_rows = []
for target, color in [(10, "#b33"), (25, "#e08a1e"), (50, "#1856a7")]:
    curve = cash_curve(target)
    money = curve["деньги нарастающим итогом, ₽"]
    ax.plot(curve["месяц"], money / 1000, color=color, label=f"цель {target} студий")
    positive = curve[money >= 0]
    cash_rows.append({"цель, студий": target, "нужно денег на старт, ₽": -money.min(),
                      "самая глубокая точка, месяц": int(curve.loc[money.idxmin(), "месяц"]),
                      "выход в плюс нарастающим итогом, месяц": int(positive["месяц"].iloc[0]) if len(positive) else "нет за 36 мес",
                      "деньги к 36 месяцу, ₽": money.iloc[-1]})
ax.axhline(0, color="#888", lw=0.8)
ax.set(xlabel="месяц", ylabel="тыс. ₽ нарастающим итогом",
       title=f"кассовый разрыв при цене {INTERNAL['price_episode_rub']:.0f} ₽ за выпуск и {INTERNAL['ramp_per_month']} новых студиях в месяц")
ax.legend(); fig.tight_layout(); fig.savefig(WORK / "internal_cash.png", dpi=160); plt.show()
cash_table = pd.DataFrame(cash_rows)
display(cash_table.round(0))
cash_table.to_csv(WORK / "internal_cash.csv", index=False, encoding="utf-8-sig")
display(Markdown(
    f"при цене {INTERNAL['price_episode_rub']:.0f} ₽ за выпуск одна студия приносит **{rub(margin_internal)} ₽** маржи в месяц "
    f"({100 * margin_internal / price_internal:.0f}% подписки), LTV/CAC **{ltv / acq_one:.1f}**. "
    f"Постоянные затраты — **{rub(fixed_month)} ₽/мес**. Минимальная постоянная база для покрытия операций — "
    f"**{minimum_studios(breakeven_ops)}**. Для покрытия операций, вложений {rub(invest)} ₽ и первого привлечения "
    f"за {h} мес. — **{minimum_studios(breakeven_full)}**. Годовая выручка при 50 студиях — "
    f"около {50 * price_internal * 12 / 1e6:.1f} млн ₽."))

как мы бы проверили это на пилоте: 12 новых записей как минимум трёх шоу, шесть выпусков делаем руками, шесть с инструментом, распределение заранее, потом редакторы меняются. один редактор не размечает одну и ту же запись обоими способами, иначе он её запомнит и второй раз сделает быстрее. фиксируем активные минуты на главы, описание, исправление спикеров, число правок и число выдуманных фактов. главное сравнение: разница минут на час аудио с поправкой на редактора и шоу. 12 записей это бюджет пилота, а не расчёт мощности, дисперсии времени у нас пока нет.

ориентиры, которые мы ставим себе заранее: медиана активной работы не больше 35 минут на часовой выпуск, ни одного выдуманного факта в принятом описании и положительный чистый эффект в базовом сценарии. это наши проектные решения, и достигнутыми мы их пока не называем. если эффект окажется отрицательным или качество провалится, дорабатываем и собираем новый независимый test.

**Вывод по блоку.** При A = 0,5 модель затрат даёт 45,5 минуты редактора на часовой выпуск вместо 75. Для студии это 16,0 высвобожденного часа в месяц, 0,100 FTE и 10 851 ₽/мес. чистого ресурсного эффекта. При вложениях 120 000 ₽ расчётная окупаемость — 11,1 месяца, ROI за два года — 57,7%. Равномерная нарезка даёт 56,0 минуты по той же формуле. Разница 10,5 минуты зависит от числа предложенных глав, пропусков и цен правок. Преимущество по F1 осталось у равномерной нарезки. Если ручная подготовка занимает 50 минут, чистый эффект становится отрицательным (−2 691 ₽/мес.). При цене пропуска 15 минут окупаемость растёт до 54,9 месяца. Для проверки нужны хронометраж и оценки редакторов.

**Сервис для нескольких студий.** В упрощённой модели подписка в 30% от стоимости высвобожденного времени составляет 4 797 ₽/мес., маржа до постоянных расходов и замены ушедших клиентов — 2 622 ₽/мес. При 10, 25 и 50 студиях ROI за 24 месяца отрицателен. В расширенной модели тариф 230 ₽ за выпуск даёт 7 475 ₽/мес. с базовой студии, маржу 4 347 ₽/мес. (58%), LTV/CAC = 4,02 и окупаемость привлечения 8,3 месяца. Для покрытия операций нужно минимум 12 постоянно обслуживаемых студий, для возврата вложений и первого привлечения за 24 месяца при постоянной базе — 31. Это другие условия, чем постепенный набор клиентов. При подключении до трёх студий в месяц и целевой базе 25 клиентов максимальный кассовый разрыв — 1,21 млн ₽, накопленный баланс выходит в плюс на 36-м месяце. Для цели в 50 клиентов — 1,40 млн ₽ и 28-й месяц. При цели в 10 клиентов баланс отрицателен на всём горизонте 36 месяцев. Готовность студий платить эту цену ещё предстоит проверить.

## 11. живая демонстрация: главы, поиск по теме и новый файл

сначала показываем результат первого test-выпуска из этого прогона: его можно открыть и послушать. поиск по темам и есть тематическая навигация. форма ниже прогоняет новую запись через весь пайплайн с методом, выбранным на dev: распознавание, спикеры, главы, рубрики и названия от Qwen.

имена спикеров вводятся вручную, модель личности не распознаёт.

**продуктовый интерфейс.** то, как сервис увидит продюсер, команда вынесла на отдельный сайт: [Chapterly](https://podcast-chapters-mvp.vercel.app/). там можно загрузить файл, дать ссылку на YouTube или вставить текст, указать длительность и число глав и получить главы с таймкодами и готовый блок для описания видео. сейчас сайт работает в облегчённом режиме MVP: расшифровку нужно вставить вручную, главы собираются на TF-IDF, а автоматическое распознавание подключается отдельным GPU-воркером, которого пока нет. поэтому полный пайплайн с выбранным ASR (в этом прогоне Whisper), Nemotron и Qwen работает только в этом ноутбуке, а сайт показывает, как будет выглядеть продукт. следующий шаг после пилота это GPU-воркер, который принимает файл с сайта и возвращает результат этого пайплайна.

In [ ]:
# показываем результат первого test-выпуска: главы, рубрики, спикеры и описание
demo_id = exp.ids["test"][0]
demo_chapters = exp.generations[(demo_id, "predicted")]
display(pd.DataFrame(demo_chapters)[["start", "title", "topic", "speakers", "summary"]].assign(start=lambda d: d.start.map(timestamp)))
print(exp.descriptions[demo_id])
from html import escape

def show_navigation(chapters, audio_url, output_path=None):
    # собираем html-таблицу глав с поиском по теме и ссылкой на нужный момент аудио
    rows = "".join(f'<tr><td>{timestamp(c["start"])}</td><td>{escape(c["title"])}</td><td>{escape(c.get("topic", ""))}</td>'
                   f'<td><a href="{escape(audio_url)}#t={int(c["start"])}" target="_blank">Слушать фрагмент</a></td></tr>' for c in chapters)
    # поиск работает прямо в браузере: скрываем строки, где нет введённого текста
    content = ('<meta charset="utf-8"><style>body{font-family:system-ui;max-width:1100px;margin:30px auto}td,th{padding:12px;border-bottom:1px solid #dde2ea;text-align:left}table{border-collapse:collapse;width:100%}input{padding:12px;width:90%;margin-bottom:20px}</style>'
               '<h2>Тематическая навигация по выпуску</h2><input id="q" placeholder="Поиск по теме или названию" oninput="document.querySelectorAll(&quot;tbody tr&quot;).forEach(r=>r.hidden=!r.textContent.toLowerCase().includes(this.value.toLowerCase()))">'
               '<table><thead><tr><th>Время</th><th>Глава</th><th>Тема</th><th>Аудио</th></tr></thead><tbody>'+rows+'</tbody></table>')
    # сохраняем страницу в файл, чтобы её можно было открыть отдельно
    if output_path: Path(output_path).write_text(content, encoding="utf-8")
    display(HTML(content))

show_navigation(demo_chapters, exp.episodes[demo_id]["audio_url"], WORK / "outputs" / str(demo_id) / "navigation.html")
# если в форме указан свой источник, сразу прогоняем через продукт и его
if DEMO_SOURCE.strip():
    custom_chapters, custom_description, custom_path = exp.demo_new(DEMO_SOURCE, DEMO_LIMIT_MIN)
    display(pd.DataFrame(custom_chapters)[["start", "title", "topic", "speakers"]]); print(custom_description)
    print("Результат нового выпуска:", custom_path)

In [ ]:
# форма для новой записи: ссылка или файл, имена спикеров и лимит минут
# форма работает внутри colab, публичный сервер и аккаунт сервиса нам не нужны
import ipywidgets as widgets
source_widget = widgets.Text(value="", description="Ссылка/путь:", layout=widgets.Layout(width="95%"))
upload_widget = widgets.FileUpload(accept=".mp3,.mp4,.wav,.m4a,.ogg,.flac,.webm", multiple=False, description="Выбрать файл")
names_widget = widgets.Text(value="{}", description="Имена JSON:", layout=widgets.Layout(width="95%"))
minutes_widget = widgets.IntText(value=DEMO_LIMIT_MIN, description="Лимит минут:")
run_widget = widgets.Button(description="Обработать запись", button_style="primary")
output_widget = widgets.Output()

def process_demo(_):
    # пока запись обрабатывается, блокируем кнопку, чтобы не запустить её дважды
    run_widget.disabled = True
    with output_widget:
        output_widget.clear_output()
        try:
            source = source_widget.value.strip()
            # если загрузили файл, сохраняем его в папку запуска и работаем с ним
            if upload_widget.value:
                values = upload_widget.value
                uploaded = next(iter(values.values())) if isinstance(values, dict) else values[0]
                dest = WORK / "uploads" / Path(uploaded["name"]).name
                dest.parent.mkdir(parents=True, exist_ok=True); dest.write_bytes(bytes(uploaded["content"]))
                source = str(dest)
            # имена спикеров приходят как json, например {"speaker_0": "Ведущий"}
            names = json.loads(names_widget.value or "{}")
            assert isinstance(names, dict), "Имена задаются JSON-объектом"
            assert minutes_widget.value >= 0, "Лимит не может быть отрицательным"
            print("Обрабатываю новую запись. Модели загружаются по очереди.", flush=True)
            chapters, description, directory = exp.demo_new(source, minutes_widget.value, names)
            display(pd.DataFrame(chapters)[["start", "title", "topic", "speakers", "summary"]].assign(start=lambda d: d.start.map(timestamp)))
            print(description)
            # пакуем результат в zip, чтобы продюсер мог его скачать
            import shutil
            archive = shutil.make_archive(str(directory.parent / "publication"), "zip", directory)
            display(FileLink(archive))
        except Exception as error:
            # ошибку показываем в форме, а не роняем весь ноутбук
            print(type(error).__name__ + ": " + str(error))
        finally:
            run_widget.disabled = False

run_widget.on_click(process_demo)
display(widgets.VBox([source_widget, upload_widget, names_widget, minutes_widget, run_widget, output_widget]))

**Вывод по блоку.** Для выпуска 1031 сформированы шесть глав, таблица таймкодов с рубриками и метками голосов, описание и HTML-навигация. Качество названий и рубрик редактор ещё не оценивал. Для этого выпуска ни одна из пяти предложенных внутренних границ не совпала с семью авторскими в пределах ±30 секунд. В полном прогоне показан результат для test-выпуска и создана форма загрузки. Обработка отдельной пользовательской записи через форму в этом прогоне не зафиксирована. На Tesla T4 сумма рабочих этапов составила около 7,7 минуты на час аудио без загрузки моделей.

## 12. файлы для сдачи и презентации

после расчётов ноутбук сам собирает фактические числа для защиты и структуру девяти слайдов, все прогнозы там подписаны как прогнозы. в архиве результатов лежат расшифровки и тексты по чужим выпускам, поэтому в открытый git он не идёт; в исходном коде проекта таких файлов нет.

In [ ]:
# собираем числа этого прогона для защиты в один файл
import shutil
selected = exp.summary[exp.summary.method == exp.selected].iloc[0]
uniform = exp.summary[exp.summary.method == "uniform"].iloc[0]
facts = {"status": "computed_by_this_run", "profile": RUN_PROFILE, "selected_model": exp.selected,
         "dev_ids": exp.ids["dev"], "test_ids": exp.ids["test"], "audio_hours": audio_hours,
         "f1_30": float(selected["F1@30"]), "precision_30": float(selected["P@30"]), "recall_30": float(selected["R@30"]),
         "uniform_f1_30": float(uniform["F1@30"]), "delta_mean": delta, "delta_lo": delta_lo, "delta_hi": delta_hi,
         "asr_selected": exp.asr_selected, "asr_dev": exp.asr_table.to_dict("records"), "rtf": rtf, "generation_fallback_rate": float(fallback_rate), "economics": result,
         "internal_economics": {"price_episode_rub": INTERNAL["price_episode_rub"], "price_month_rub": price_internal,
                                "margin_studio_rub": margin_internal, "ltv_cac": ltv / acq_one,
                                "breakeven_ops_studios": breakeven_ops, "breakeven_full_studios": breakeven_full,
                                "fixed_month_rub": fixed_month, "invest_rub": invest},
         "repository_url": REPOSITORY_URL.strip() or None}
save_json(WORK / "results_for_defense.json", facts)
# краткая текстовая сводка; экономику явно подписываем как прогноз
summary_text = (f"Профиль: {RUN_PROFILE}. Данные: {audio_hours:.2f} ч, dev {exp.ids['dev']}, test {exp.ids['test']}.\n"
                f"Выбор на dev: {exp.asr_selected} + {exp.selected}. Test F1@30={selected['F1@30']:.3f}; uniform={uniform['F1@30']:.3f}.\n"
                f"Парный прирост: {delta:+.3f}, 95% bootstrap [{delta_lo:+.3f};{delta_hi:+.3f}].\n"
                f"RTF production={rtf:.3f}. Отказ формата/цитаты LLM={fallback_rate:.1%}.\n"
                f"ПРОГНОЗ: было {ECON.manual_min:.1f}, стало {result['minutes_after']:.1f} мин на часовой выпуск; "
                f"{result['fte_saved']:.3f} FTE; чистый ресурсный эффект {result['net_month_rub']:,.0f} ₽/мес.\n"
                f"СЕРВИС (сценарий): {INTERNAL['price_episode_rub']:.0f} ₽ за выпуск, маржа сервиса с одной студии {margin_internal:,.0f} ₽/мес, "
                f"LTV/CAC {ltv / acq_one:.1f}, минимум студий для покрытия операций: {minimum_studios(breakeven_ops)}, для возврата вложений и первого привлечения за {h} мес. при постоянной базе: {minimum_studios(breakeven_full)}.\n"
                "Экономический эффект у заказчика, WER/DER и редакторское принятие без разметки не подтверждены.\n")
(WORK / "summary.txt").write_text(summary_text, encoding="utf-8")
# структура слайдов: заголовок и основной текст, числа подставляются из этого прогона
slides = [
    ("Черновик глав для редактора подкаста", "Заказчик: студия с 5-10 часовыми выпусками в неделю. Пользователь проверяет главы, спикеров и описание."),
    (f"Ручная подготовка занимает {result['fte_before']:.2f} FTE в базовом сценарии", f"{ECON.episodes_month:.1f} выпуска/мес × {ECON.manual_min:.0f} мин = {result['hours_before']:.1f} ч/мес. Все операционные входы - допущения."),
    ("Ошибки границ переводятся в минуты правок", f"FP стоит {ECON.remove_fp_min:g} мин, FN {ECON.find_fn_min:g} мин. Дополнительно: проверка глав, названий и описания. F1 без P/R не определяет экономику."),
    (f"Настройка на {len(exp.ids['dev'])} выпусках, проверка на {len(exp.ids['test'])}", f"Dev: {exp.ids['dev']}; test: {exp.ids['test']}. {audio_hours:.2f} ч аудио. Авторские главы - слабый эталон одного IT-шоу."),
    ("Модели загружаются по очереди и работают без API", f"ASR выбран на dev из GigaAM-v3, Whisper large-v3-turbo, Parakeet-TDT-0.6B-v3 и Qwen3-ASR-1.7B: {exp.asr_selected}. Nemotron streaming, TF-IDF/USER2/Qwen embeddings, Qwen3-4B, описание. Коммиты закреплены."),
    (f"Выбранный метод дал F1@30={selected['F1@30']:.2f}", f"Uniform={uniform['F1@30']:.2f}; разница {delta:+.3f} [{delta_lo:+.3f};{delta_hi:+.3f}]. Метод выбран по dev. Ошибки: см. error_review.csv."),
    ("Название требует независимой редакторской проверки", f"Автоматические proxy сравниваются на одинаковых интервалах. Отказ формата/цитаты LLM={fallback_rate:.1%}. Принятие A отдельно от hit@1."),
    (f"Прогноз высвобождения составляет {result['fte_saved']:.2f} FTE", f"{result['minutes_after']:.1f} мин/выпуск; чистый ресурсный эффект {result['net_month_rub']:,.0f} ₽/мес; окупаемость {result['payback_months']:.1f} мес. Как сервис: {INTERNAL['price_episode_rub']:.0f} ₽ за выпуск, минимальная постоянная база для покрытия операций — {minimum_studios(breakeven_ops)}, для возврата вложений и привлечения за {h} мес. — {minimum_studios(breakeven_full)}. У заказчика пока не измерено."),
    ("Демонстрация и следующий пилот", "Ноутбук показывает главы, спикеров, описание и навигацию для test-выпуска. Запуск формы для новой записи и пилот с редакторами проверяются отдельно."),
]
slide_text = "# Слайды, 10-15 минут\n\n" + "\n\n".join(
    f"## Слайд {i+1}. {title}\n\n{body}" for i, (title, body) in enumerate(slides))
(WORK / "defense_slides.md").write_text(slide_text, encoding="utf-8")
print(summary_text)
print("Ссылка на репозиторий:", REPOSITORY_URL.strip() or "ещё не указана")

# исходники кладём отдельно от результатов, а в отдельном файле помечаем, что из результатов нельзя публиковать
source_out = WORK / "source"; source_out.mkdir(exist_ok=True)
for name, source in SOURCE_FILES.items(): (source_out / name).write_text(source, encoding="utf-8")
save_json(WORK / "publication_warning.json", {"do_not_publish": ["cache/", "outputs/", "custom/", "uploads/", "human_review.csv", "human_review_template.csv", "title_proxy_metrics.csv", "error_review.csv", "experiment_manifest.json"],
          "reason": "Аудио и производные тексты чужих выпусков не входят в открытый репозиторий"})
print("Готовы summary.txt, results_for_defense.json, defense_slides.md и outputs/.")

## 13. сохранение результатов

в архив попадают таблицы, графики, исходники и полные результаты для внутренней проверки. веса моделей и аудиокэш туда не кладём, они слишком большие и легко скачиваются заново.

In [ ]:
# собираем в один zip таблицы, графики, исходники и результаты; веса моделей и аудиокэш не берём
import zipfile
archive_path = WORK.parent / f"podcast_results_{RUN_PROFILE}.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in WORK.rglob("*"):
        # пропускаем кэш, загрузки пользователя и служебные папки python
        if path.is_file() and not any(part in {"cache", "uploads", "__pycache__"} for part in path.relative_to(WORK).parts):
            archive.write(path, path.relative_to(WORK))
display(FileLink(str(archive_path)))
# если в форме включено скачивание, отдаём архив сразу
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))

**вывод по блоку.** после прогона у нас есть все файлы для сдачи: сводка, числа для защиты, структура слайдов и архив с таблицами и графиками. расшифровки и тексты по выпускам «Радио-Т» доступны в рабочей среде и полном архиве. Для публичного репозитория отобраны исходный код, агрегированные показатели, числовые таблицы и графики.

## 14. ограничения ноутбука и MVP

здесь в одном месте собраны все ограничения, которые разбросаны по разделам выше. их стоит держать в голове, когда читаете числа, и не обещать заказчику больше, чем мы проверили.

**данные и эталон**

- в эксперименте шесть выпусков одного IT-шоу «Радио-Т» (14.65 часа): два на dev, четыре на test. интервью, бренд-подкасты и другие жанры мы не проверяли, поэтому перенос результата на них не доказан.
- эталон это авторские главы ведущих. они субъективны и отмечают не каждую смену темы, так что часть «ошибок» модели может быть спорной, а часть настоящих границ эталон просто не содержит.
- профиль `quick` берёт по 60 минут из четырёх выпусков и нужен только для проверки запуска, выводы по нему делать нельзя.

**качество модели**

- На test TF-IDF получил macro F1@30 = 0,036 против 0,113 у равномерной нарезки. Парная разница −0,078, 95%-й бутстреп-интервал по четырём выпускам от −0,144 до −0,012. Единственное попадание в допуск ±30 секунд пришлось на выпуск 1033.
- Предсказанные границы лежат на сетке с шагом 30 секунд, минимальная глава — 90 секунд. Граница может совпасть с эталоном точнее, чем на 30 секунд.
- WER распознавания и DER диаризации на подкастах не измерены: нет ручной расшифровки и разметки спикеров. цифры из ДЗ1 получены на других данных и годятся только для выбора модели.
- стресс-тест портит текст искусственно (удаления и замены слов), это не настоящие ошибки распознавания на плохом звуке.
- качество названий оценено только автоматическими прокси (косинус и hit@1 против авторского названия). долю названий, которые редактор принимает без правок (A), никто не мерил; в экономике это сценарии 0.25, 0.5 и 0.75. качество рубрик не измерено совсем.
- Для 27 найденных глав доля запасных названий — 0%. Автоматическая проверка подтверждает соответствие формату и порогу совпадения слов в цитате. Факты, имена и точность пересказа требуют редакторской проверки.
- описание для YouTube собирается шаблоном из названий глав, отдельной проверки фактов в нём нет, его должен прочитать редактор.

**спикеры и язык**

- Качество выбранного Whisper на английских вставках, терминах и именах отдельно не оценивалось.
- Nemotron различает не больше восьми голосов, на перекрывающейся речи доли говорящих приблизительные.
- модель не знает, кто говорит: метки `speaker_0`, `speaker_1` продюсер переименовывает вручную, а если голосов много или они похожи, метки нужно проверять.

**техника и демо**

- Зафиксированный полный прогон на Tesla T4 в Colab занял около 2 ч 36 мин от начала первой ячейки до конца последней, включая подготовку, сравнение кандидатов и оценку. На четырёх test-выпусках тёплый RTF рабочих этапов равен 0,129. Скорость обработки новой записи или на CPU здесь не измеряли.
- модели загружаются по очереди на каждую новую запись, поэтому обработка занимает минуты, а одновременно стоит обрабатывать одну запись. очереди задач, личных кабинетов и нескольких пользователей нет.
- сайт [Chapterly](https://podcast-chapters-mvp.vercel.app/) пока работает в облегчённом режиме: расшифровку вставляют вручную, главы собираются на TF-IDF, распознавания и спикеров там нет. полный пайплайн запускается только в ноутбуке, и загруженная в форму запись лежит во временной среде Colab и пропадает после её отключения.
- поправить границу или название прямо в ноутбуке или на сайте нельзя, правки делаются после скачивания. интеграции с YouTube, CMS студии и монтажными программами нет, описание копируется вручную.
- загрузка с YouTube из Colab иногда блокируется, тогда нужен файл или прямая ссылка на mp3.

**права и персональные данные**

- у «Радио-Т» лицензия CC BY-NC-ND 3.0, поэтому аудио, расшифровки и сгенерированные по ним тексты в открытый репозиторий не кладём, а коммерческий продукт на этих записях строить нельзя. нужны записи самого заказчика и права на них.
- Голоса и содержание записей могут позволять идентифицировать участников. В учебном MVP отсутствуют пользовательские согласия, правила доступа и сроки хранения. Перед пилотом студии нужно определить условия обработки, место хранения и порядок удаления записей гостей.

**экономика**

- живого интервью со студией не было: 75 минут ручной работы, 7.5 выпуска в неделю, 1000 ₽ за час редактора и коэффициенты правок (FN 10 минут, FP 0.5 минуты) это допущения, а не хронометраж. При исходных 50 минутах ручной работы сценарный чистый эффект отрицателен (−2 691 ₽/мес.).
- высвобожденные часы становятся деньгами, только если редактора переключают на другую работу или нанимают меньше людей.
- цена 230 ₽ за выпуск, CAC, отток 3% в месяц, стоимость GPU, сервера, бухгалтерии и темп подключения студий в расчёте сервиса это сценарии компании. готовность студий платить такую цену не проверена.
- 50 студий по данным РАЭК за 2023 год это ориентир, а не TAM и не список клиентов; в расширенной модели при постоянной базе и тарифе 230 ₽ за выпуск расчётный порог возврата вложений за 24 месяца — 31 студия. Из этой оценки нельзя вывести достижимую долю рынка.

большую часть пунктов про качество и экономику закрывает пилот, описанный в разделе 10: новые записи трёх шоу, хронометраж редакторов, ручная оценка названий и разметка спикеров на нескольких фрагментах. ограничения про права, персональные данные и многопользовательскую работу закрываются уже разработкой продукта, а не экспериментом.

## 15. вопросы и вклад команды

1. **почему F1@30 и при чём тут экономика?** 30 секунд это нормальная точность для навигации по выпуску, а устойчивость мы проверяем ещё на 15 и 60 секундах. в деньги переводим не сам F1, а отдельно FP, FN и число глав: два решения с одинаковым F1 могут стоить редактору совсем разного времени.
2. **где могла быть утечка?** названия и число глав test-выпусков модель не видит и при настройке не используется. Параметры IDF для границ и выбор метода определены на dev. названия отдельно оцениваем на эталонных интервалах, чтобы не смешивать их с ошибками сегментации.
3. **Почему на test использован Whisper?** На двух dev-выпусках среди четырёх ASR-кандидатов Whisper вместе с TF-IDF дал лучший macro F1@30 глав (0,417). GigaAM — исходный кандидат из ДЗ1. Ручной расшифровки этих выпусков нет, поэтому сравнивать кандидатов по WER на них пока невозможно.
4. **почему ROI пока прогнозный?** коэффициенты правок, стоимость часа и принятие названий у заказчика ещё не измерены. из эксперимента мы берём реальные ошибки и RTF, а экономическая модель показывает, как результат зависит от остальных входов.
5. **что мешает внедрению?** (полный список ограничений в разделе 14) маленький test из одного шоу, нет разметки для WER и DER, субъективные авторские главы, риск неточных аннотаций и неизвестный объём ручных правок. пилот на трёх шоу нужен как раз чтобы проверить эти места.

| участник | что сделал(а) |
| --- | --- |
| Еремина Анастасия | по итогам своего ДЗ1 предложила GigaAM-v3 для сравнения ASR и Nemotron для диаризации; предложила, как перевести ошибки границ глав в минуты продюсера; написала выводы по блокам |
| Бровин Данила | перенёс из своего ДЗ сценарий с пользовательскими видео и zero-shot определение темы, на этой идее построены рубрики; собрал выпуски «Радио-Т» с авторскими главами и разбил их на dev и test; запускал полный прогон в Colab |
| Майлычко Анастасия | подключила GigaAM с нарезкой silero VAD и сделала сегментацию на эмбеддингах; подобрала параметры на dev и посчитала метрики качества; провела стресс-тест и разбор ошибок |
| Мошкович Герман | сделал генерацию названий и кратких описаний глав через Qwen3-4B с проверкой цитаты; подготовил шаблон ручной оценки названий; написал описания блоков и декларацию ИИ |
| Семенова Аксинья | посчитала экономику: FTE до и после, затраты, окупаемость, ROI и чувствительность; собрала живое демо с формой для новой записи; подготовила слайды для защиты |

### декларация ИИ и заимствований

при подготовке проекта мы пользовались ChatGPT (OpenAI): разбор требований, переработка архитектуры и кода, формулы, тесты, пояснения и материалы для защиты. часть ранних наработок, код и тексты, сделана с Claude (Anthropic), это отмечено в исходном `repo/podcast_chapters_mvp.ipynb`. с помощью Claude также сделаны комментарии в коде, подключение GigaAM-v3 и USER2-base, сравнение четырёх моделей распознавания (GigaAM-v3, Whisper large-v3-turbo, Parakeet-TDT-0.6B-v3, Qwen3-ASR-1.7B), порог полезности в экономике (стоимость пропущенной главы), мягкая проверка цитат в генерации, ускоренный подсчёт WER в стресс-тесте, расчёт внутренней экономики компании (коридор цены, P&L, кассовый разрыв) и тексты описаний в ноутбуке. измеренные показатели в выводах сверены с полным прогоном. Экономические прогнозы и цели пилота подписаны отдельно.

из `ДЗ1_Еремина.ipynb` (ДЗ1 Ереминой) мы взяли GigaAM-v3 и способ её загрузки, идеи оценки ASR, разделения WER и DER, VAD и потоковой обработки. из `asr_benchmark_user_video_topic.ipynb` (ДЗ Бровина) сценарий с пользовательским видео и тематизацию. из папок `podcast-chapters` и `repo` продуктовую постановку, подход TextTiling, набор артефактов и структуру экономики. код при этом переписан: точное сопоставление границ, IDF только по dev, формула с FP и FN, отдельная ручная оценка названий, нейтральные метки спикеров и честный учёт отказов LLM. старые цифры из других работ мы этой работе не приписываем.

### источники

- Hearst M. A. (1997). TextTiling: Segmenting Text into Multi-paragraph Subtopic Passages. https://aclanthology.org/J97-1003/
- Pevzner L., Hearst M. A. (2002). A Critique and Improvement of an Evaluation Metric for Text Segmentation. https://aclanthology.org/J02-1002/
- Сбер (ai-sage). GigaAM-v3: карточка модели, ветки ctc, rnnt, e2e. https://huggingface.co/ai-sage/GigaAM-v3
- deepvk. USER2-base: префиксы задач и mean pooling. https://huggingface.co/deepvk/USER2-base
- Silero Team. silero-vad. https://github.com/snakers4/silero-vad
- NVIDIA. Nemotron-3-Diarization: streaming API и speaker cache, лицензия и ограничение в восемь голосов. https://huggingface.co/nvidia/Nemotron-3-Diarization
- Qwen. Qwen3-Embedding-0.6B. https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
- Qwen. Qwen3-4B-Instruct-2507. https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507
- «Радио-Т». RSS и страницы №1029-1034. https://radio-t.com/podcast.rss
- РАЭК (2023). Около 10 миллионов россиян активно слушают подкасты. https://www.raec.ru/live/branch/14160/
- YouTube Help. Video Chapters: 00:00, минимум три главы, не короче 10 секунд. https://support.google.com/youtube/answer/9884579

веб-источники и коммиты моделей проверены 19.11.2026.

### Итоговый вывод

Полный прогон на Tesla T4 обработал 14,65 часа «Радио-Т» и подготовил для четырёх test-выпусков расшифровки, 27 глав с названиями и метками голосов, рубрики и описания. По двум dev-выпускам выбраны Whisper large-v3-turbo и TF-IDF. На test качество границ снизилось: macro F1@30 составил 0,036 против 0,113 у равномерной нарезки. Лучший результат в сравнении на test у Qwen3-Embedding (0,255). Проверять его как новую основную модель нужно на других выпусках.

При сценарном принятии половины названий прогноз работы редактора — 45,5 минуты вместо 75, чистый ресурсный эффект — 10 851 ₽/мес. Практическую пользу пока предстоит измерить: для пилота нужны новые записи нескольких шоу, независимые оценки названий и хронометраж редакторов.